# 📋 Notebook 03 — Training & Evaluation

## Mục tiêu

Train và so sánh 2 mô hình phân loại **Real vs Fake**:
1. **Random Forest** — ensemble of decision trees, dễ giải thích
2. **XGBoost** — gradient boosting, thường cho accuracy cao hơn

### Metrics đánh giá
- **Accuracy**: tỉ lệ dự đoán đúng tổng thể
- **Precision / Recall / F1**: chi tiết cho từng class
- **ROC-AUC**: khả năng phân biệt 2 class ở mọi ngưỡng
- **Confusion Matrix**: ma trận nhầm lẫn
- **Feature Importance**: features nào đóng góp nhiều nhất
- **Per-generator accuracy**: model yếu ở generator nào?

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.train import (
    train_random_forest, train_xgboost, evaluate_model,
    plot_confusion_matrix, plot_roc_curve, plot_feature_importance
)

# Load data
FEATURE_DIR = r"E:\ai_image_detector\features"
MODEL_DIR = r"E:\ai_image_detector\models"
os.makedirs(MODEL_DIR, exist_ok=True)

X_train = np.load(os.path.join(FEATURE_DIR, "X_train.npy"))
y_train = np.load(os.path.join(FEATURE_DIR, "y_train.npy"))
X_test = np.load(os.path.join(FEATURE_DIR, "X_test.npy"))
y_test = np.load(os.path.join(FEATURE_DIR, "y_test.npy"))

with open(os.path.join(FEATURE_DIR, "feature_names.txt")) as f:
    feature_names = f.read().strip().split("\n")

test_meta = pd.read_csv(os.path.join(FEATURE_DIR, "test_meta.csv"))

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Features: {len(feature_names)}")
print(f"Train: {sum(y_train==0)} real, {sum(y_train==1)} fake")
print(f"Test:  {sum(y_test==0)} real, {sum(y_test==1)} fake")

## Phần 1: Train Random Forest

Random Forest tạo nhiều decision trees, mỗi tree được train trên subset ngẫu nhiên của data. Kết quả cuối = vote đa số.

**Hyperparameter search** bằng GridSearchCV + 5-fold cross-validation.

In [ ]:
# === 1.1 Train Random Forest ===
print("Training Random Forest...")
rf_model, rf_grid = train_random_forest(X_train, y_train, cv=5)

# Evaluate
rf_metrics = evaluate_model(rf_model, X_test, y_test, "Random Forest")

## Phần 2: Train XGBoost

XGBoost (eXtreme Gradient Boosting) xây dựng cây quyết định tuần tự — mỗi cây mới cố gắng **sửa lỗi** của cây trước đó.

In [ ]:
# === 2.1 Train XGBoost ===
print("Training XGBoost...")
xgb_model, xgb_grid = train_xgboost(X_train, y_train, cv=5)

# Evaluate
xgb_metrics = evaluate_model(xgb_model, X_test, y_test, "XGBoost")

## Phần 3: So sánh 2 mô hình

Trực quan so sánh: Confusion Matrix, ROC Curve, Feature Importance.

In [ ]:
# === 3.1 So sánh trực quan ===

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# Row 1: Confusion Matrix + ROC
plot_confusion_matrix(y_test, rf_metrics["y_pred"], "Random Forest", ax=axes[0, 0])
plot_confusion_matrix(y_test, xgb_metrics["y_pred"], "XGBoost", ax=axes[0, 1])

# ROC cả 2 model trên 1 chart
plot_roc_curve(y_test, rf_metrics["y_proba"], "Random Forest", ax=axes[0, 2])
plot_roc_curve(y_test, xgb_metrics["y_proba"], "XGBoost", ax=axes[0, 2])

# Row 2: Feature importance
plot_feature_importance(rf_model, feature_names, top_n=15, ax=axes[1, 0])
axes[1, 0].set_title("Random Forest — Top 15 Features")

plot_feature_importance(xgb_model, feature_names, top_n=15, ax=axes[1, 1])
axes[1, 1].set_title("XGBoost — Top 15 Features")

# Bảng so sánh metrics
axes[1, 2].axis('off')
comparison_text = f"""
SO SÁNH 2 MÔ HÌNH
{'─' * 35}

              Random Forest    XGBoost
Accuracy      {rf_metrics['accuracy']:.4f}          {xgb_metrics['accuracy']:.4f}
ROC-AUC       {rf_metrics['roc_auc']:.4f}          {xgb_metrics['roc_auc']:.4f}

Best model: {'XGBoost' if xgb_metrics['accuracy'] > rf_metrics['accuracy'] else 'Random Forest'}
"""
axes[1, 2].text(0.1, 0.5, comparison_text, fontsize=13, fontfamily='monospace',
                verticalalignment='center', transform=axes[1, 2].transAxes,
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))

plt.suptitle("So sánh Random Forest vs XGBoost", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## Phần 4: Phân tích per-generator

Kiểm tra model có yếu ở generator nào không — nếu accuracy thấp ở 1 generator cụ thể, có thể cần thêm data hoặc features khác.

In [ ]:
# === 4.1 Per-generator accuracy ===
from sklearn.metrics import accuracy_score

# Chọn best model
best_name = "XGBoost" if xgb_metrics['accuracy'] > rf_metrics['accuracy'] else "Random Forest"
best_model = xgb_model if xgb_metrics['accuracy'] > rf_metrics['accuracy'] else rf_model
best_pred = xgb_metrics['y_pred'] if xgb_metrics['accuracy'] > rf_metrics['accuracy'] else rf_metrics['y_pred']

print(f"Phân tích per-generator cho model: {best_name}")
print("=" * 55)

generators = test_meta["generator"].unique()
gen_acc = {}

for gen in sorted(generators):
    mask = test_meta["generator"] == gen
    if mask.sum() == 0:
        continue
    acc = accuracy_score(y_test[mask], best_pred[mask])
    gen_acc[gen] = acc
    n = mask.sum()
    correct = (y_test[mask] == best_pred[mask]).sum()
    print(f"  {gen:12s}: accuracy = {acc:.4f}  ({correct}/{n})")

# Chart
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(gen_acc.keys(), gen_acc.values(), color='steelblue', edgecolor='white')
ax.set_ylabel("Accuracy")
ax.set_title(f"Per-generator Accuracy ({best_name})")
ax.set_ylim(0, 1.05)
ax.axhline(y=best_model.__class__.__name__ and np.mean(list(gen_acc.values())), 
           color='red', linestyle='--', label='Mean accuracy')
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, gen_acc.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', fontsize=9)

plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Phần 5: Lưu model tốt nhất

In [ ]:
# === 5.1 Lưu best model ===

model_path = os.path.join(MODEL_DIR, "model.pkl")
joblib.dump(best_model, model_path)
print(f"✓ Best model ({best_name}) đã lưu: {model_path}")

# Lưu cả RF lẫn XGB để tham khảo
joblib.dump(rf_model, os.path.join(MODEL_DIR, "random_forest.pkl"))
joblib.dump(xgb_model, os.path.join(MODEL_DIR, "xgboost.pkl"))
print(f"✓ Cả 2 model đã lưu trong {MODEL_DIR}/")

# Tóm tắt
print(f"\n{'=' * 55}")
print(f"  TÓM TẮT")
print(f"{'=' * 55}")
print(f"  Random Forest: accuracy={rf_metrics['accuracy']:.4f}, AUC={rf_metrics['roc_auc']:.4f}")
print(f"  XGBoost:       accuracy={xgb_metrics['accuracy']:.4f}, AUC={xgb_metrics['roc_auc']:.4f}")
print(f"  Best model:    {best_name}")
print(f"  Saved to:      {model_path}")